In [2]:
import random
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules


Generate Synthetic Transactions in a small market scenario

In [3]:
# acquire dataset
df = pd.read_csv("market_dataset.csv", index_col=0)
df


,beer,bread,coffee,diapers,milk,water
0,True,True,True,False,True,True
1,True,False,True,True,False,False
2,False,False,False,False,True,False
3,False,False,False,True,False,False
4,True,False,False,False,True,False
...,...,...,...,...,...,...
295,False,False,False,False,False,False
296,True,True,False,False,False,True
297,False,False,False,False,False,False
298,False,True,False,False,True,False


In [4]:
# get frequent itemset
frequent_itemsets = apriori(df, min_support=0.07, use_colnames=True)
frequent_itemsets.sort_values("support", ascending=False).head(10)


,support,itemsets
1,0.463333,(bread)
4,0.396667,(milk)
0,0.366667,(beer)
5,0.353333,(water)
2,0.283333,(coffee)
3,0.223333,(diapers)
13,0.200000,"(bread, milk)"
6,0.176667,"(bread, beer)"
8,0.173333,"(beer, diapers)"
14,0.166667,"(bread, water)"


In [5]:
# calculate association rules

In [6]:
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)
rules.sort_values(["confidence", "lift"], ascending=False).head(10)


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
21,"(bread, diapers)",(beer),0.096667,0.366667,0.083333,0.862069,2.351097,1.0,0.047889,4.591667,0.636162,0.219298,0.782214,0.544671
33,"(milk, diapers)",(beer),0.093333,0.366667,0.073333,0.785714,2.142857,1.0,0.039111,2.955556,0.588235,0.189655,0.661654,0.492857
5,(diapers),(beer),0.223333,0.366667,0.173333,0.776119,2.116689,1.0,0.091444,2.828889,0.679267,0.416000,0.646504,0.624423
28,"(milk, beer)",(bread),0.153333,0.463333,0.083333,0.543478,1.172975,1.0,0.012289,1.175556,0.174173,0.156250,0.149338,0.361667
9,(milk),(bread),0.396667,0.463333,0.200000,0.504202,1.088205,1.0,0.016211,1.082429,0.134346,0.303030,0.076152,0.467928
40,"(milk, water)",(bread),0.153333,0.463333,0.076667,0.500000,1.079137,1.0,0.005622,1.073333,0.086614,0.141975,0.068323,0.332734
1,(beer),(bread),0.366667,0.463333,0.176667,0.481818,1.039895,1.0,0.006778,1.035673,0.060576,0.270408,0.034444,0.431557
22,"(beer, diapers)",(bread),0.173333,0.463333,0.083333,0.480769,1.037631,1.0,0.003022,1.033580,0.043871,0.150602,0.032489,0.330313
32,"(milk, beer)",(diapers),0.153333,0.223333,0.073333,0.478261,2.141467,1.0,0.039089,1.488611,0.629563,0.241758,0.328233,0.403310
4,(beer),(diapers),0.366667,0.223333,0.173333,0.472727,2.116689,1.0,0.091444,1.472989,0.832996,0.416000,0.321108,0.624423


In [8]:
import plotly.io as pio
pio.renderers.default = 'notebook'


In [37]:
import plotly.express as px

#rules_plot = rules.sort_values("confidence", ascending=False).head(15).copy()
rules_plot = rules.sort_values("confidence", ascending=False).copy()

# Convert frozensets to strings for Plotly
rules_plot["antecedents_str"] = rules_plot["antecedents"].apply(lambda x: ", ".join(list(x)))
rules_plot["consequents_str"] = rules_plot["consequents"].apply(lambda x: ", ".join(list(x)))


fig = px.scatter(
    rules_plot,
    x="support",
    y="confidence",
    size="lift",
    #hover_name="antecedents_str",
    hover_data={
        "antecedents_str": True,
        "consequents_str": True,
        "support": ":.3f",
        "confidence": ":.3f",
        "lift": ":.3f"
    },
    title="Association Rules: Support vs Confidence (Bubble size = Lift)",
    width=800, height=600
)

fig.update_layout(
    xaxis_title="Support",
    yaxis_title="Confidence",
    template="plotly_white"
)

fig.show()


In [35]:
import plotly.express as px

#rules_plot = rules.sort_values("confidence", ascending=False).head(15).copy()
rules_plot = rules.sort_values("confidence", ascending=False).copy()

# Convert frozensets to strings for Plotly
rules_plot["antecedents_str"] = rules_plot["antecedents"].apply(lambda x: ", ".join(list(x)))
rules_plot["consequents_str"] = rules_plot["consequents"].apply(lambda x: ", ".join(list(x)))
rules_plot["antecedents_len"] = rules_plot["antecedents"].apply(len)
rules_plot["consequents_len"] = rules_plot["consequents"].apply(len)
rules_plot["rule_len"] = rules_plot["consequents_len"] + rules_plot["antecedents_len"]


fig = px.scatter(
    rules_plot,
    x="support",
    y="confidence",
    size="lift",
    color="rule_len",
    #hover_name="antecedents_str",
    color_continuous_scale=["grey", "cyan"],
    hover_data={
        "antecedents_str": True,
        "consequents_str": True,
        "support": ":.3f",
        "confidence": ":.3f",
        "lift": ":.3f"
    },
    title="Association Rules: Support vs Confidence (Bubble size = Lift, color= #elements in the rule)",
    width=800, height=600
)

fig.update_layout(
    xaxis_title="Support",
    yaxis_title="Confidence",
    template="plotly_white"
)

fig.show()


## Comments:

From the results, we can observe how "diapers" + "beer" have high lift, high support and high confidence, possibly indicating an interesting underlying phenomenon (you are a dad!)